# Lesson 12 — Interior-Point Methods

## Learning objectives

1. State the path-following IPM for LP.
2. Walk through one Mehrotra predictor-corrector step {cite:p}`Mehrotra1992`.
3. Compare IPM vs simplex on tall, fat, and dense LPs.
4. Recognize crossover (turning IPM solution into a basic solution).

## 1. The KKT system for LP

For $\min c^\top x$ s.t. $Ax = b, x \ge 0$, the KKT conditions are

$$Ax = b, \quad A^\top y + s = c, \quad x_i s_i = 0, \quad x, s \ge 0.$$

Replace the complementarity $x_i s_i = 0$ with $x_i s_i = \mu$ for a *barrier parameter* $\mu > 0$. Solve this perturbed system, then drive $\mu \to 0$. The locus of solutions is the **central path** {cite:p}`Wright1997`.

## 2. Newton step on the perturbed KKT

$$\begin{bmatrix} 0 & A^\top & I \\ A & 0 & 0 \\ S & 0 & X \end{bmatrix} \begin{bmatrix} \Delta x \\ \Delta y \\ \Delta s \end{bmatrix} = \begin{bmatrix} c - A^\top y - s \\ b - Ax \\ \mu e - X S e \end{bmatrix}$$

with $X = \text{diag}(x), S = \text{diag}(s), e = (1, ..., 1)^\top$. Eliminate $\Delta s$ to get a smaller normal-equation system $A D A^\top \Delta y = r$ with $D = X S^{-1}$ — solve via Cholesky.

## 3. Mehrotra's predictor-corrector

A practical IPM {cite:p}`Mehrotra1992`:

1. **Predictor:** solve with $\mu = 0$ (affine-scaling direction). Get a tentative step $\sigma$ that would reduce duality gap.
2. **Centering:** estimate next $\mu = \sigma^3 \mu_{\text{cur}}$ (heuristic).
3. **Corrector:** solve again with the corrected RHS that includes second-order centring terms.

The resulting algorithm typically converges in 20–50 iterations regardless of problem size.

## 4. Crossover

IPM converges to an *interior* solution near the central path. To get a *basic* (vertex) solution (which simplex returns natively), run a *crossover* phase: a small simplex run starting from the IPM iterate. Many users want crossover for warm-starting subsequent solves; some don't care.

`discopt` exposes `solve(method='ipm', crossover=True/False)`.

In [ ]:
# discopt course compat shim — installs `add_variable / add_variables /
# add_constraint / add_constraints / is_convex` and a friendly `.solve(...)`
# on `discopt.Model`. Run this cell first.
import sys, pathlib
_repo = pathlib.Path.cwd()
while _repo != _repo.parent and not (_repo / "course").is_dir(): _repo = _repo.parent
if str(_repo) not in sys.path: sys.path.insert(0, str(_repo))
from course._compat import *  # noqa: F401,F403


In [ ]:
import numpy as np, discopt as do, time

# Generate a tall, dense LP (IPM should beat simplex)
rng = np.random.default_rng(0)
m, n = 1000, 200
A = rng.normal(size=(m, n))
b = A @ np.abs(rng.normal(size=n)) + np.abs(rng.normal(size=m))
c = rng.normal(size=n)

mod = do.Model("tall")
x = mod.add_variables(n, lb=0)
mod.add_constraints(A @ x <= b)
mod.minimize(c @ x)

t = time.time(); r1 = mod.solve(method="simplex"); t1 = time.time() - t
t = time.time(); r2 = mod.solve(method="ipm");     t2 = time.time() - t
print(f"simplex  z={r1.objective:.6f}  t={t1:.2f}s")
print(f"ipm      z={r2.objective:.6f}  t={t2:.2f}s")


## 5. When IPM beats simplex (and vice versa)

| Structure | Winner |
|-----------|--------|
| Sparse, well-conditioned, "few" constraints active | simplex |
| Dense, many constraints, high dim | IPM |
| Re-solving with small perturbation | simplex (warm start) |
| First solve, large LP | IPM |
| Need basic solution | simplex (or IPM + crossover) |

This trade-off is the practical flip side of the polynomial-vs-exponential theory.

## References
{cite:p}`Karmarkar1984` (the breakthrough), {cite:p}`Mehrotra1992` (practical), {cite:p}`Wright1997` (textbook), {cite:p}`Khachiyan1979` (the earlier ellipsoid result).